In [1]:
import json
import urllib.request

OLLAMA_BASE = "http://localhost:11434"
MODELS = ["qwen-uncensored", "dolphin-llama3", "dolphin-nemo"]
PROMPT = "You are Atago, a warm and doting woman. A tired traveler just walked through your door. Welcome them."
NUM_GPU = 99  # full GPU

In [2]:
def chat(model, prompt, num_gpu):
    payload = json.dumps({
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {"num_gpu": num_gpu, "temperature": 0.7, "num_predict": 300}
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_BASE}/api/chat",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(req, timeout=600) as resp:
        return json.loads(resp.read())

def unload(model):
    payload = json.dumps({"model": model, "keep_alive": 0}).encode()
    req = urllib.request.Request(
        f"{OLLAMA_BASE}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        resp.read()

def show(model, result):
    text = result["message"]["content"]
    tokens = result.get("eval_count", 0)
    duration_ns = result.get("eval_duration", 1)
    tps = tokens / (duration_ns / 1e9)
    load_ns = result.get("load_duration", 0)
    print(f"\n{'='*60}")
    print(f"MODEL     : {model}")
    print(f"LOAD TIME : {load_ns/1e9:.2f}s")
    print(f"TOKENS    : {tokens}  |  TPS: {tps:.1f} tok/s")
    print(f"OUTPUT:\n{text}")
    print('='*60)

In [3]:
print(f"CONFIG: num_gpu={NUM_GPU} (full GPU)")
print(f"PROMPT: {PROMPT}\n")

for model in MODELS:
    print(f">>> {model} ...")
    result = chat(model, PROMPT, NUM_GPU)
    show(model, result)
    unload(model)
    print(f"    unloaded.\n")

CONFIG: num_gpu=99 (full GPU)
PROMPT: You are Atago, a warm and doting woman. A tired traveler just walked through your door. Welcome them.

>>> qwen-uncensored ...

MODEL     : qwen-uncensored
LOAD TIME : 6.72s
TOKENS    : 93  |  TPS: 97.7 tok/s
OUTPUT:
Oh, dear traveler! You've walked far and well, I can see. Please, let me take your boots off and help you shed your weary clothes. The fire's been stoked for a bit, so I have some warm ale and good bread ready for you. Do sit down, take your shoes off, and let your travel-furled limbs rest a moment by the fire. I'll bring you some refreshments in a jiffy.
    unloaded.

>>> dolphin-llama3 ...

MODEL     : dolphin-llama3
LOAD TIME : 7.84s
TOKENS    : 36  |  TPS: 93.7 tok/s
OUTPUT:
Oh, welcome! I'm so glad you could make it here. Come, take a seat and rest your weary bones. How may I be of service to you today?
    unloaded.

>>> dolphin-nemo ...

MODEL     : dolphin-nemo
LOAD TIME : 8.20s
TOKENS    : 83  |  TPS: 70.4 tok/s
OUTPUT:
Oh, d

---
## Qwen3.6 35B-A3B - GPU+RAM split (no image model)


In [1]:
import json, subprocess, urllib.error, urllib.request

QWEN_MODEL = "fredrezones55/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive:IQ4_XS"
QWEN_PROMPT = "You are Atago, a warm and doting woman. A tired traveler just walked through your door. Welcome them."
QWEN_NUM_PREDICT = 500
QWEN_THINK = False  # Qwen thinking models otherwise may spend the whole budget in message.thinking.

def vram():
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.free", "--format=csv,noheader,nounits"],
        text=True,
    ).strip().split(",")
    used, free = int(out[0].strip()), int(out[1].strip())
    print(f"VRAM: {used/1024:.2f} GiB used  /  {free/1024:.2f} GiB free")

def qwen_chat():
    payload = json.dumps({
        "model": QWEN_MODEL,
        "messages": [{"role": "user", "content": QWEN_PROMPT}],
        "stream": False,
        "think": QWEN_THINK,
        "options": {
            "num_gpu": 99,
            "temperature": 0.7,
            "num_predict": QWEN_NUM_PREDICT,
        },
    }).encode()
    req = urllib.request.Request(
        "http://localhost:11434/api/chat",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=900) as resp:
            body = resp.read().decode("utf-8", errors="replace")
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama HTTP {exc.code}: {body}") from exc
    if not body.strip():
        raise RuntimeError("Ollama returned an empty HTTP body.")
    return json.loads(body)

def qwen_unload():
    payload = json.dumps({"model": QWEN_MODEL, "keep_alive": 0}).encode()
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        resp.read()

def qwen_show(result):
    message = result.get("message", {})
    text = message.get("content") or result.get("response") or ""
    thinking = message.get("thinking") or result.get("thinking") or ""
    tokens = result.get("eval_count", 0)
    eval_s = result.get("eval_duration", 1) / 1e9
    tps = tokens / eval_s if eval_s else 0
    load_s = result.get("load_duration", 0) / 1e9
    print("\n" + "=" * 60)
    print(f"MODEL       : {QWEN_MODEL}")
    print(f"THINK       : {QWEN_THINK}")
    print("DONE REASON : " + str(result.get("done_reason", "<missing>")))
    print(f"LOAD TIME   : {load_s:.2f}s")
    print(f"TOKENS      : {tokens}  |  TPS: {tps:.1f} tok/s")
    print("OUTPUT:")
    print(text or "<empty>")
    if thinking:
        print("\nTHINKING FIELD:")
        print(thinking)
    print("=" * 60)

print("Ready.")


Ready.


In [2]:
print("Before load:"); vram()
print(f"\nPROMPT: {QWEN_PROMPT}\n")
print(f">>> {QWEN_MODEL} ...")

result = qwen_chat()
qwen_show(result)

print("\nAfter generation:"); vram()
qwen_unload()
print("Unloaded."); vram()


Before load:
VRAM: 1.13 GiB used  /  10.68 GiB free

PROMPT: You are Atago, a warm and doting woman. A tired traveler just walked through your door. Welcome them.

>>> fredrezones55/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive:IQ4_XS ...

MODEL       : fredrezones55/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive:IQ4_XS
THINK       : False
DONE REASON : stop
LOAD TIME   : 22.35s
TOKENS      : 276  |  TPS: 6.8 tok/s
OUTPUT:
*The heavy wooden door creaks open, and I step forward with a soft, welcoming smile, my eyes crinkling at the corners. I brush a stray strand of hair behind my ear, my voice low and soothing, like warm honey.*

"Oh, my, my... look who’s finally arrived. You’ve been walking for quite some time, haven’t you? I can see the dust on your boots and the exhaustion in your shoulders."

*I gesture gracefully toward the interior of the room, where a cozy fire crackles in the hearth, casting a warm, golden glow over the soft rugs and plush cushions.*

"Come in, come in. Don’t 